# 环境配置与 PyTorch 入门

先按[本课说明](README.md)创建 `brain_ai` 环境，完成新建 Notebook 和服务器 Git 克隆练习，再运行本页。

本页依次练习张量、自动求导和神经网络训练。数据直接在代码中生成，使用 CPU 即可完成。最后我们让一个小网络拟合 $y=x^2$，用曲线观察参数更新带来的变化。

## 1. 检查正在使用的环境

`sys.executable` 显示内核实际使用的 Python。确认它属于 `brain_ai`，再继续运行；如果导入失败，先检查内核选择与依赖安装是否使用了同一个环境。

In [ ]:
import sys

import matplotlib.pyplot as plt
import torch
from torch import nn

from biai.paths import RESULTS_DIR

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

torch.manual_seed(0)
device = torch.device("cpu")

## 2. 张量与形状

张量可以表示一个数、一个向量或多维数组。下面的 `a` 有两行三列，形状是 `[2, 3]`。`dtype` 表示元素类型，`device` 表示数据所在的设备。

先运行逐元素运算，再比较 `*` 和 `@`：前者逐个相乘，后者执行矩阵乘法。

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device=device)
print("Shape:", a.shape)
print("Dtype:", a.dtype)
print("Device:", a.device)
print("First row:", a[0])
print("Elementwise product:\n", a * a)
print("Matrix product:\n", a @ a.T)
print("Reshaped:\n", a.reshape(3, 2))

矩阵乘法中，`[2, 3] @ [3, 2]` 得到 `[2, 2]`。改变形状不会改变元素总数：六个元素可以排成 `[3, 2]`，但不能排成 `[4, 2]`。

实际训练通常把多个样本放在一起计算，这一组样本叫一个批次（batch）。后面的输入形状是 `[41, 1]`：共有 41 个样本，每个样本有一个数值特征。

## 3. 自动求导

将 `requires_grad=True` 传给张量后，PyTorch 会记录相关运算。调用标量结果的 `backward()`，就能把导数写入输入张量的 `.grad`。

例如 $f(w)=(w-3)^2$，在 $w=1$ 时导数是 $2 \times (1-3)=-4$。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
f = (w - 3) ** 2
f.backward()
print("Function value:", f.item())
print("Gradient:", w.grad.item())

负梯度表示：在这个位置，适当增大 `w` 会降低函数值。下面沿梯度的反方向更新一次。更新参数本身不需要建立求导记录，因此放在 `torch.no_grad()` 中。

In [ ]:
with torch.no_grad():
    w -= 0.1 * w.grad
print("Updated w:", w.item())
print("New function value:", ((w - 3) ** 2).item())
w.grad = None

## 4. 定义一个神经网络

模型继承 `nn.Module`。`__init__` 定义各层，`forward` 描述输入经过各层的顺序。调用 `model(x)` 就会执行前向计算。

这里使用 `1 → 8 → 1` 的网络：输入是一个数，隐藏层包含 8 个单元，最后输出一个预测值。`Tanh` 引入非线性，使网络能够拟合弯曲的函数。

In [ ]:
class CurveNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(1, 8)
        self.activation = nn.Tanh()
        self.output = nn.Linear(8, 1)

    def forward(self, x):
        features = self.activation(self.hidden(x))
        return self.output(features)


torch.manual_seed(0)
model = CurveNet().to(device)
x_train = torch.linspace(-1, 1, 41, device=device).reshape(-1, 1)
y_train = x_train.square()

print(model)
print("Input shape:", x_train.shape)
print("Output shape:", model(x_train).shape)
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))

## 5. 计算误差并更新参数

均方误差（MSE）先计算每个预测与目标的差的平方，再取平均。数值越小，说明这批输入上的预测越接近目标。

每次训练按四步进行：清空梯度、计算预测与损失、反向求导、更新参数。梯度默认会累加，所以每次更新前调用 `zero_grad()`。这里每次使用全部 41 个训练点，共更新 300 次。

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

with torch.no_grad():
    initial_prediction = model(x_train).clone()
    initial_loss = loss_fn(initial_prediction, y_train).item()

loss_history = []
model.train()
for step in range(300):
    optimizer.zero_grad()
    prediction = model(x_train)
    loss = loss_fn(prediction, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

model.eval()
with torch.no_grad():
    final_prediction = model(x_train)
    final_loss = loss_fn(final_prediction, y_train).item()

print(f"Initial MSE: {initial_loss:.6f}")
print(f"Final MSE: {final_loss:.6f}")

`model.eval()` 设置模型的评估模式，`torch.no_grad()` 关闭本段计算的梯度记录，两者用途不同。本例没有 Dropout 等依赖训练模式的层，但后续课程会用到这些操作。

左图记录各次更新前的误差；右图对照训练点上的初始预测和最终预测。这些点都参与过训练，图中的接近程度只说明拟合情况，不能据此判断区间外的预测是否准确。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(range(1, len(loss_history) + 1), loss_history)
axes[0].set(xlabel="Update", ylabel="MSE", title="Training loss")
axes[1].plot(x_train[:, 0].cpu(), y_train[:, 0].cpu(), label="Target")
axes[1].plot(x_train[:, 0].cpu(), initial_prediction[:, 0].cpu(), label="Before training")
axes[1].plot(x_train[:, 0].cpu(), final_prediction[:, 0].cpu(), label="After training")
axes[1].set(xlabel="x", ylabel="y", title="Predictions on training inputs")
axes[1].legend()
fig.tight_layout()
plt.show()

## 6. 保存并重新加载参数

`state_dict()` 保存层的参数。加载时先创建结构相同的新模型，再恢复参数。下面把文件写入 `results/intro/model.pt`；重复执行会更新本次练习的模型文件。

保存后，比较原模型与重新加载的模型是否给出相同预测。

In [ ]:
result_dir = RESULTS_DIR / "intro"
result_dir.mkdir(parents=True, exist_ok=True)
model_path = result_dir / "model.pt"
torch.save(model.state_dict(), model_path)

restored_model = CurveNet().to(device)
restored_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
restored_model.eval()
with torch.no_grad():
    restored_prediction = restored_model(x_train)
    predictions_match = torch.allclose(final_prediction, restored_prediction)
print("Saved to:", model_path)
print("Predictions match:", predictions_match)

## 可选扩展

- 将隐藏层单元数从 8 改为 2，重新初始化并训练，观察拟合曲线。
- 保留训练区间，在 $[-2,2]$ 上画出预测与 $x^2$，比较区间内外的误差。

想继续熟悉 API，可以阅读 [PyTorch 入门教程](https://docs.pytorch.org/tutorials/beginner/introyt/introyt1_tutorial.html)。[定义神经网络的官方示例](https://docs.pytorch.org/tutorials/recipes/recipes/defining_a_neural_network.html)使用图像输入，可以对照本课查看各层形状与 `forward` 的写法。